In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random

In [2]:
fake = Faker()
random.seed(42)

In [3]:
# 1. CUSTOMERS (400)

customers = []
for i in range(1, 401):
    customers.append([
        i,
        fake.name(),
        random.choice(["Male", "Female"]),
        random.randint(18, 60),
        fake.city(),
        fake.state(),
        fake.date_between(start_date="-3y", end_date="today")
    ])

customers_df = pd.DataFrame(customers, columns=[
    "customer_id", "customer_name", "gender",
    "age", "city", "state", "signup_date"
])


In [4]:
# 2. CATEGORIES (10)

categories = [
    (1, "Electronics"), (2, "Clothing"), (3, "Footwear"),
    (4, "Home Appliances"), (5, "Books"),
    (6, "Beauty"), (7, "Sports"),
    (8, "Toys"), (9, "Groceries"), (10, "Furniture")
]

categories_df = pd.DataFrame(categories, columns=[
    "category_id", "category_name"
])


In [5]:
# 3. PRODUCTS (300)

products = []
for i in range(1, 301):
    products.append([
        i,
        fake.word().capitalize(),
        random.randint(1, 10),
        round(random.uniform(100, 5000), 2)
    ])

products_df = pd.DataFrame(products, columns=[
    "product_id", "product_name", "category_id", "unit_price"
])


In [6]:
# 4. ORDERS (2000)
# -----------------------
orders = []
for i in range(1, 2001):
    orders.append([
        i,
        random.randint(1, 400),
        fake.date_between(start_date="-2y", end_date="today"),
        random.choice(["UPI", "Credit Card", "Debit Card", "Cash", "Net Banking"]),
        random.choice(["Completed", "Cancelled", "Returned"])
    ])

orders_df = pd.DataFrame(orders, columns=[
    "order_id", "customer_id", "order_date",
    "payment_method", "order_status"
])


In [7]:
# 5. ORDER ITEMS (~3000)
# -----------------------
order_items = []
order_item_id = 1

for order_id in orders_df["order_id"]:
    for _ in range(random.randint(1, 3)):
        product_id = random.randint(1, 300)
        price = float(products_df.loc[
            products_df["product_id"] == product_id, "unit_price"
        ])
        quantity = random.randint(1, 5)

        order_items.append([
            order_item_id,
            order_id,
            product_id,
            quantity,
            price
        ])
        order_item_id += 1

order_items_df = pd.DataFrame(order_items, columns=[
    "order_item_id", "order_id", "product_id",
    "quantity", "price"
])


C:\Users\chiranjeevi\AppData\Local\Temp\ipykernel_18052\347860799.py:9: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(products_df.loc[


In [11]:
# SAVE FILES
# -----------------------
customers_df.to_csv("customers.csv", index=False)
categories_df.to_csv("categories.csv", index=False)
products_df.to_csv("products.csv", index=False)
orders_df.to_csv("orders.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)

print("✅ Data generation completed successfully!")

✅ Data generation completed successfully!


In [16]:
import pandas as pd
from sqlalchemy import create_engine

# -----------------------
# MYSQL CONNECTION
# -----------------------
USER = "root"
PASSWORD = "root"        # change if needed
HOST = "localhost"
PORT = "3306"
DATABASE = "sales_etl1"

engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}",
    echo=True
)

In [17]:
customers_df = pd.read_csv("customers.csv")
categories_df = pd.read_csv("categories.csv")
products_df = pd.read_csv("products.csv")
orders_df = pd.read_csv("orders.csv")
order_items_df = pd.read_csv("order_items.csv")


In [18]:
# -----------------------
# LOAD TABLES
# -----------------------

customers_df.to_sql(
    name="customers",
    con=engine,
    if_exists="replace",
    index=False
)

categories_df.to_sql(
    name="categories",
    con=engine,
    if_exists="replace",
    index=False
)

products_df.to_sql(
    name="products",
    con=engine,
    if_exists="replace",
    index=False
)

orders_df.to_sql(
    name="orders",
    con=engine,
    if_exists="replace",
    index=False
)

order_items_df.to_sql(
    name="order_items",
    con=engine,
    if_exists="replace",
    index=False
)

print("✅ Data successfully loaded into MySQL")


2026-02-03 13:06:12,705 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-02-03 13:06:12,706 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 13:06:12,710 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-02-03 13:06:12,711 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 13:06:12,713 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-02-03 13:06:12,714 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 13:06:12,716 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 13:06:12,719 INFO sqlalchemy.engine.Engine DESCRIBE `sales_etl1`.`customers`
2026-02-03 13:06:12,720 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 13:06:12,724 INFO sqlalchemy.engine.Engine 
CREATE TABLE customers (
	customer_id BIGINT, 
	customer_name TEXT, 
	gender TEXT, 
	age BIGINT, 
	city TEXT, 
	state TEXT, 
	signup_date TEXT
)


2026-02-03 13:06:12,724 INFO sqlalchemy.engine.Engine [no key 0.00061s] {}
2026-02-03 13:06:12,794 INFO sqlalchemy.engine.Engine INSERT INTO 